# ChatUniTest LoRA Fine-tuning

**Goal**: Fine-tune Qwen2.5-Coder-7B-Instruct with QLoRA to generate Java JUnit tests

**Before running**:
- Make sure your HuggingFace token (write access) is ready

## Step 1: Install Dependencies

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%pip install -q transformers==4.44.0 peft==0.11.1 trl==0.9.6 \
    "bitsandbytes>=0.44.0" accelerate==0.33.0 \
    datasets sentencepiece huggingface_hub

## Step 2: Login to HuggingFace (write token required)

For our training to work, you will need to connect to HuggingFace with a token that has write access.

In [ ]:
from huggingface_hub import login, notebook_login

# Option 1
notebook_login()

# Option 2
# login(token="hf_xxxxxxxxxxxxx")

## Step 3: Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import re
from dry_run_config import DRY_RUN_CONFIG, print_dry_run_summary

dry_run_config = DRY_RUN_CONFIG
dataset_config = dry_run_config["dataset"]
print_dry_run_summary()

PROMPT_TEMPLATE = """mode=COMPLETION
projectPath=unknown
assertionStyle=JUNIT
staticSnapshot:
{context}
runtimeFacts:

### JUnit Test:
"""

def build_full_text(context: str, test: str) -> str:
    return PROMPT_TEMPLATE.format(context=context.strip()) + test.strip()

def has_assertion(test: str) -> bool:
    return any(kw in test for kw in ["assert", "Assert", "verify", "Verify", "fail("])

def is_valid_java(code: str) -> bool:
    return code.count("{") > 0 and abs(code.count("{") - code.count("}")) <= 2

def estimate_tokens(text: str) -> int:
    return len(text) // 4

# Load dataset
print("Loading dataset...")
raw = load_dataset(dataset_config["dataset_name"], split=dataset_config["split"])
print(f"Raw samples: {len(raw)}")
print(f"Columns: {raw.column_names}")

# Auto-detect column names
context_column = next((c for c in ["context", "input", "source"] if c in raw.column_names), None)
test_column = next((t for t in ["test", "output", "target"] if t in raw.column_names), None)
print(f"Using: context='{context_column}', test='{test_column}'")

df = raw.to_pandas()[[context_column, test_column]].copy()
df.columns = ["context", "test"]

# Data cleaning
before = len(df)
df = df.drop_duplicates(subset=["context"])
print(f"After dedup: {len(df)} (removed {before - len(df)})")

df = df[df["test"].apply(has_assertion)]
print(f"After assertion filter: {len(df)}")

df = df[df["test"].apply(is_valid_java)]
print(f"After Java structure filter: {len(df)}")

df = df[df["context"].str.strip().str.len() > 30]
print(f"After empty context filter: {len(df)}")

# Build full training text
df["text"] = df.apply(lambda row: build_full_text(row["context"], row["test"]), axis=1)
df["token_est"] = df["text"].apply(estimate_tokens)
df = df[df["token_est"] <= dataset_config["max_seq_length"]]
print(f"After token length filter: {len(df)}")

# Sample dry-run subset
df = df.sample(frac=1, random_state=42).reset_index(drop=True).head(dataset_config["max_samples"])
print(f"\nFinal training samples: {len(df)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(df[["text"]])

# Preview first sample
print("\n=== Sample preview (first 500 chars) ===")
print(train_dataset[0]["text"][:500])

## Step 4: Load Base Model (QLoRA 4-bit)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from dry_run_config import DRY_RUN_CONFIG

dry_run_config = DRY_RUN_CONFIG
base_model_name = dry_run_config["base_model"]
quantization_config_values = dry_run_config["quantization"]

dtype_map = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
compute_dtype = dtype_map.get(quantization_config_values["bnb_4bit_compute_dtype"], torch.bfloat16)

# 4-bit QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=quantization_config_values["load_in_4bit"],
    bnb_4bit_use_double_quant=quantization_config_values["bnb_4bit_use_double_quant"],
    bnb_4bit_quant_type=quantization_config_values["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=compute_dtype,
 )

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"Model loaded. VRAM usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 5: Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from dry_run_config import DRY_RUN_CONFIG

lora_config_values = DRY_RUN_CONFIG["lora"]

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=lora_config_values["r"],
    lora_alpha=lora_config_values["lora_alpha"],
    target_modules=lora_config_values["target_modules"],
    lora_dropout=lora_config_values["lora_dropout"],
    bias=lora_config_values["bias"],
    task_type=lora_config_values["task_type"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6: Train
Replace `hf_username` with your HuggingFace username before running this step.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from dry_run_config import DRY_RUN_CONFIG

training_config = DRY_RUN_CONFIG["training"]
dataset_config = DRY_RUN_CONFIG["dataset"]

# Set your HuggingFace username here
hf_username = "your-hf-username"   # <-- REPLACE THIS
output_model_name = f"{hf_username}/my-testgen-lora-light"

training_args = TrainingArguments(
    output_dir="./my-testgen-lora-light",
    num_train_epochs=training_config["num_train_epochs"],
    max_steps=training_config["max_steps"],
    per_device_train_batch_size=training_config["per_device_train_batch_size"],
    gradient_accumulation_steps=training_config["gradient_accumulation_steps"],
    gradient_checkpointing=training_config["gradient_checkpointing"],
    learning_rate=training_config["learning_rate"],
    lr_scheduler_type=training_config["lr_scheduler_type"],
    warmup_ratio=training_config["warmup_ratio"],
    fp16=training_config["fp16"],
    bf16=training_config["bf16"],
    logging_steps=training_config["logging_steps"],
    save_strategy=training_config["save_strategy"],
    save_steps=training_config["save_steps"],
    save_total_limit=training_config["save_total_limit"],
    report_to=training_config["report_to"],
    optim=training_config["optim"],
    max_grad_norm=training_config["max_grad_norm"],
    group_by_length=training_config["group_by_length"],
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=dataset_config["max_seq_length"],
    packing=False,
)

print("Starting training...")
print(
    f"Total steps: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}"
)
trainer.train()

## Step 7: Save and Push to HuggingFace Hub

In [ ]:
print("Saving model...")
trainer.save_model("./my-testgen-lora-light")

print(f"Pushing to HuggingFace Hub: {output_model_name}")
model.push_to_hub(output_model_name, private=False)
tokenizer.push_to_hub(output_model_name, private=False)

print(f"\nDone! Model available at: https://huggingface.co/{output_model_name}")
print(f"\nNext step — update model_server.py line 23:")
print(f'  PeftModel.from_pretrained(model, "{output_model_name}")')

## Step 8 (Optional): Resume from Checkpoint

If Colab disconnects, use this cell to resume from the latest checkpoint.

In [ ]:
# import os

# # Find latest checkpoint
# checkpoints = [
#     d for d in os.listdir("./my-testgen-lora-light")
#     if d.startswith("checkpoint-")
# ]
# if checkpoints:
#     latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
#     resume_from = f"./my-testgen-lora-light/{latest}"
#     print(f"Resuming from {resume_from}")
#     trainer.train(resume_from_checkpoint=resume_from)
# else:
#     print("No checkpoint found — please run from Step 1")